# 03 - LHA ES and IMCC

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [IMA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: scenario P&L cubes with scenario metadata, risk-factor definitions, RFET real-price observations and evidence, stress-history series, PLA/backtesting observations, and NMRF valuation artifacts. The package dataset contract defines the committed fixture files, manifest lineage, and Arrow handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged workflow is described in the [IMA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook reviews the vector-based liquidity-horizon adjustment and IMCC decomposition for the committed `capital_run_v1` fixture. It uses the shared fixture loader in `examples.capital_run_fixture`; no independent scenario data is created here.

Regulatory anchors: Basel MAR33 expected shortfall and liquidity-horizon adjustment concepts; U.S. NPR 2.0 proposed-rule parameters for nested liquidity-horizon ES and the unconstrained/constrained IMCC blend cited below; EU CRR Articles 325bc and 325bd as comparison context.

Prototype caution: scenario values are synthetic and positive = loss. Outputs are deterministic model-validation evidence for this prototype, not final regulatory capital.


## Public API happy path

This notebook builds liquidity-horizon adjusted expected shortfall inputs and computes IMCC with top-level IMA APIs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import load_capital_run_v1_fixture, policy_from_fixture
from frtb_ima import (
    LiquidityHorizon,
    ModellabilityStatus,
    ScenarioCube,
    imcc_breakdown_for_policy,
    nested_lh_vectors_from_cube,
    per_risk_class_nested_lh_vectors_from_cube,
    risk_factor_names_for_lh_subset,
    route_nmrf_classifications_for_capital,
)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


def filtered_cube(cube: ScenarioCube, risk_factor_names: tuple[str, ...]) -> ScenarioCube:
    allowed = set(risk_factor_names)
    selected = tuple(name for name in cube.risk_factor_names if name in allowed)
    if not selected:
        raise ValueError(f"no requested risk factors found in ScenarioCube '{cube.name}'")
    indices = [cube.risk_factor_index[name] for name in selected]
    values = cube.values[:, :, indices].copy()
    values.flags.writeable = False
    return ScenarioCube(
        values=values,
        scenario_metadata=cube.scenario_metadata,
        position_ids=cube.position_ids,
        risk_factor_names=selected,
        name=f"{cube.name}_imcc_population",
    )


def golden_value(name: str) -> float:
    return float(fixture.expected_outputs["scalars"][name]["value"])


def golden_check(name: str, actual: float) -> tuple[float, float, str]:
    spec = fixture.expected_outputs["scalars"][name]
    expected = float(spec["value"])
    tolerance = float(spec.get("atol", 1e-9)) + float(spec.get("rtol", 1e-9)) * abs(expected)
    diff = actual - expected
    return expected, diff, "PASS" if abs(diff) <= tolerance else "CHECK"


def tail_summary(values: np.ndarray, alpha: float) -> tuple[int, float, float]:
    sorted_desc = np.sort(np.asarray(values, dtype=float))[::-1]
    tail_count = max(1, int(np.ceil(sorted_desc.size * (1.0 - alpha))))
    tail_cutoff = float(sorted_desc[tail_count - 1])
    tail_mean = float(np.mean(sorted_desc[:tail_count]))
    return tail_count, tail_cutoff, tail_mean


classifications = {
    name: ModellabilityStatus(status)
    for name, status in fixture.expected_outputs["classifications"].items()
}
routing = route_nmrf_classifications_for_capital(classifications, policy)
risk_factor_by_name = {risk_factor.name: risk_factor for risk_factor in fixture.risk_factors}

imcc_cube = filtered_cube(fixture.scenario_cube, routing.imcc_risk_factors)
imcc_risk_factors = tuple(risk_factor_by_name[name] for name in imcc_cube.risk_factor_names)
all_risk_class_vectors = nested_lh_vectors_from_cube(
    imcc_cube,
    imcc_risk_factors,
    lha_weights=policy.lha_weights,
)
per_risk_class_vectors = per_risk_class_nested_lh_vectors_from_cube(
    imcc_cube,
    imcc_risk_factors,
    lha_weights=policy.lha_weights,
)
imcc_result = imcc_breakdown_for_policy(
    all_risk_class_vectors,
    per_risk_class_vectors,
    policy,
    run_id=str(fixture.params["run_id"]),
    desk_id=str(fixture.params["desk_id"]),
)

scenario_count = imcc_result.unconstrained.scenario_count
scenario_count_display = scenario_count if scenario_count is not None else "n/a"

display(
    Markdown(
        f"Loaded `{fixture.root.name}` for `{policy.regime.value}`. "
        f"IMCC population: `{len(routing.imcc_risk_factors)}` risk factors; "
        f"scenario count: `{scenario_count_display}`."
    )
)


## Implementation details / math verification - LHA vectors and ES tails

The remaining cells inspect nested liquidity-horizon vectors, ES tails, and IMCC reconciliation records.


## IMCC Population and Golden Reconciliation

This notebook starts after RFET classification. It uses the fixture's committed classification results only to route risk factors: modellable factors and Type A NMRFs feed IMCC, while Type B NMRFs are excluded from IMCC and handled through SES in the capital assembly. The RFET evidence narrative is intentionally left to notebook 01.


In [ ]:
routing_rows = [
    ["Modellable", str(len(routing.modellable_risk_factors)), ", ".join(routing.modellable_risk_factors)],
    ["Type A NMRF", str(len(routing.type_a_nmrf_risk_factors)), ", ".join(routing.type_a_nmrf_risk_factors)],
    ["Type B NMRF", str(len(routing.type_b_nmrf_risk_factors)), ", ".join(routing.type_b_nmrf_risk_factors)],
    ["IMCC population", str(len(routing.imcc_risk_factors)), ", ".join(routing.imcc_risk_factors)],
]
display(markdown_table(["Population", "Count", "Risk factors"], routing_rows))

reconciliation_rows = []
for label, name, actual in [
    ("Unconstrained LHA ES", "unconstrained_lha_es", imcc_result.unconstrained_lha_es),
    ("Constrained LHA ES", "constrained_lha_es", imcc_result.constrained_lha_es),
    ("IMCC", "imcc", imcc_result.imcc),
]:
    expected, diff, status = golden_check(name, actual)
    reconciliation_rows.append(
        [label, f"{actual:,.6f}", f"{expected:,.6f}", f"{diff:,.3e}", status]
    )

display(
    markdown_table(
        ["Scalar", "Notebook", "Expected output", "Delta", "Status"],
        reconciliation_rows,
    )
)


## Nested Liquidity-Horizon Vectors

The liquidity-horizon adjustment is built from nested scenario vectors, not from a scalar rescaling of one final ES value. `LH10` contains the full IMCC population; longer-horizon vectors contain only risk factors whose assigned liquidity horizon is at least that cutoff.


In [ ]:
inventory_rows: list[list[str]] = []
for liquidity_horizon, weight in policy.lha_weights:
    vector = all_risk_class_vectors.get(liquidity_horizon)
    names = risk_factor_names_for_lh_subset(imcc_risk_factors, liquidity_horizon)
    inventory_rows.append(
        [
            liquidity_horizon.name,
            str(liquidity_horizon.value),
            f"{weight:.1f}",
            str(0 if vector is None else vector.values.size),
            str(len(names)),
            ", ".join(names) if names else "omitted",
        ]
    )

display(
    markdown_table(
        ["LH cutoff", "Days", "Weight", "Scenarios", "Risk factor count", "Risk factors"],
        inventory_rows,
    )
)


## ES Tail and LHA Contributions

The left chart shows the all-class `LH10` loss distribution and its ES tail. The right chart decomposes the vector-based LHA formula into weighted squared ES terms by nested liquidity horizon.


In [ ]:
lh10_vector = all_risk_class_vectors[LiquidityHorizon.LH10]
tail_count, tail_cutoff, tail_mean = tail_summary(lh10_vector.values, policy.es_confidence_level)
components = [component for component in imcc_result.unconstrained.components if component.present]
labels = [component.liquidity_horizon.name for component in components]
weighted_squares = np.asarray([component.weighted_square for component in components], dtype=float)
total_weighted_square = float(np.sum(weighted_squares))
shares = (
    weighted_squares / total_weighted_square
    if total_weighted_square > 0.0
    else np.zeros_like(weighted_squares)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].hist(lh10_vector.values, bins=32, color="#4c78a8", alpha=0.82)
axes[0].axvline(tail_cutoff, color="#111111", linestyle="--", linewidth=1.5, label="Tail cutoff")
axes[0].axvline(tail_mean, color="#c44e52", linewidth=1.8, label="ES tail mean")
axes[0].set_title(f"LH10 loss distribution, tail count = {tail_count}")
axes[0].set_xlabel("Scenario loss, positive = loss")
axes[0].set_ylabel("Scenario count")
axes[0].legend(frameon=False)

axes[1].bar(labels, weighted_squares, color="#59a14f", alpha=0.86)
for index, share in enumerate(shares):
    axes[1].text(index, weighted_squares[index], f"{share:.0%}", ha="center", va="bottom")
axes[1].set_title("Unconstrained LHA weighted ES-squared terms")
axes[1].set_xlabel("Nested liquidity horizon")
axes[1].set_ylabel("Weight x ES^2")
fig.tight_layout()


## IMCC Blend

The constrained component sums risk-class LHA ES values without cross-class diversification credit. The final prototype IMCC is the policy-weighted blend of the all-class unconstrained result and the constrained result.


In [ ]:
risk_labels = [component.risk_class.value for component in imcc_result.constrained_components]
risk_values = np.asarray([component.lha_es for component in imcc_result.constrained_components], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].bar(["ALL"], [imcc_result.unconstrained_lha_es], color="#4c78a8", label="Unconstrained")
axes[0].bar(risk_labels, risk_values, color="#f28e2b", alpha=0.88, label="Constrained components")
axes[0].set_title("All-class versus per-risk-class LHA ES")
axes[0].set_ylabel("LHA ES")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(frameon=False)

blend_labels = [
    f"{imcc_result.unconstrained_weight:.1f} x unconstrained",
    f"{imcc_result.constrained_weight:.1f} x constrained",
    "IMCC",
]
blend_values = np.asarray(
    [
        imcc_result.unconstrained_weight * imcc_result.unconstrained_lha_es,
        imcc_result.constrained_weight * imcc_result.constrained_lha_es,
        imcc_result.imcc,
    ],
    dtype=float,
)
axes[1].bar(blend_labels[:2], blend_values[:2], color=["#4c78a8", "#f28e2b"], alpha=0.88)
axes[1].bar([blend_labels[2]], [blend_values[2]], color="#c44e52", alpha=0.9)
axes[1].set_title("Policy blend into IMCC")
axes[1].set_ylabel("Capital measure")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()


## See also

- [IMA package journey](../docs/PACKAGE_JOURNEY.md)
- [IMA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
